# 6 Foundation models, and how to decide whether to believe one

Every other lane in this course has a pretrained encoder: text has sentence embedders,
images have ResNet and DINOv2. Timeseries now has them too —
`google/timesfm-2.5-200m-transformers` loads on the `transformers` version this course
already depends on, forecasts **zero-shot** (no training on your data at all), and it is
going to beat everything the last four notebooks trained.

That headline is the least useful thing in this notebook, and you should not take it
home.

Here is why. Timeseries foundation models are pretrained on enormous mixtures of public
series, and ETTh1 — the dataset you have used for four notebooks — is one of the most
published timeseries in existence. If it is in the pretraining mixture, "zero-shot" is
a category error: the model is not generalising to your data, it is remembering it. And
you generally cannot find out, because most model cards do not say.

So this notebook is about the *procedure*, not the number:

1. get the zero-shot number, honestly;
2. **check the pretraining corpus before believing it** — a habit that transfers to
   every foundation model in every modality;
3. and test one thing that contamination cannot explain away: does the
   **freeze-the-encoder-and-cache-the-vectors** pattern — the technique the whole rest
   of this course is built on — work here? It does not, and the reason it does not is a
   rule you can apply before building anything.

The first cold run downloads about 925 MB.

In [ ]:
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import TimesFm2_5Model, TimesFm2_5ModelForPrediction

from ett import load_ett, window_index
from notebooktester import param

W, H = 512, 24        # TimesFM's patch size is 32, so 512 is exactly 16 patches
torch.manual_seed(0)

ett = load_ett()
mu, sigma = ett.values[ett.splits["train"]].mean(), ett.values[ett.splits["train"]].std()


def windows(split: str, stride: int, sample: int | None = None):
    values = (ett.values[ett.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    if sample is not None:
        win = win[::sample]
    return win[:, :W], win[:, W:]

SAMPLE = param(None, test=16)
xtr, ytr = windows("train", 2, sample=SAMPLE)
xte, yte = windows("test", 4, sample=SAMPLE)

print(f"train {tuple(xtr.shape)}   test {tuple(xte.shape)}")


def mae(y, yhat):
    return float((y - yhat).abs().mean())

## 6.1 Baselines, at this longer window

`W=512` rather than the 96 of the earlier notebooks, because that is what TimesFM's
patching wants. Re-run the baselines at the new window rather than quoting the old
numbers — they are not comparable across a window change.

In [ ]:
results = {}
results["naive"] = mae(yte, xte[:, -1:].expand(-1, H))
results["seasonal naive (24h)"] = mae(yte, xte[:, -24:][:, :H])

linear = nn.Linear(W, H)
opt = torch.optim.Adam(linear.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
t0 = time.time()
for _ in range(10):
    for xb, yb in loader:
        opt.zero_grad()
        loss_fn(linear(xb), yb).backward()
        opt.step()
linear_secs = time.time() - t0
with torch.no_grad():
    results["linear (trained)"] = mae(yte, linear(xte))

for name, score in results.items():
    print(f"{name:24s} MAE {score:.4f}   ratio to naive {score / results['naive']:.3f}")
print(f"\nlinear: {sum(p.numel() for p in linear.parameters()):,} params, {linear_secs:.1f}s")

Worth pausing on: at this window the trained linear model does **not** beat `naive`.
Five times more context than `02_basemodels` used, five times the parameters, and it
went backwards — more history is not more information when the extra history is
irrelevant and the model has to learn to ignore it. (`02_basemodels` exercise E1 asks
you to find where that turns over. This is one point on that curve.)

## 6.2 TimesFM, zero-shot

No training, no `xtr`, no `ytr`. The pretraining happened somewhere else, on somebody
else's compute, on data we did not choose.

In [ ]:
t0 = time.time()
timesfm = TimesFm2_5ModelForPrediction.from_pretrained("google/timesfm-2.5-200m-transformers").eval()
print(f"loaded in {time.time() - t0:.0f}s, {sum(p.numel() for p in timesfm.parameters()) / 1e6:.0f}M parameters")

N_ZERO_SHOT = param(256, test=1)            # decoding is slower than encoding; subsample for notebook speed
t0 = time.time()
chunks = []
with torch.no_grad():
    for i in range(0, N_ZERO_SHOT, 32):
        batch = [xte[j] for j in range(i, min(i + 32, N_ZERO_SHOT))]
        out = timesfm(past_values=batch, forecast_context_len=W)
        pred = out.mean_predictions if hasattr(out, "mean_predictions") else out[0]
        chunks.append(torch.as_tensor(pred)[:, :H])
zero_shot = torch.cat(chunks)
zero_shot_secs = time.time() - t0
n = len(zero_shot)

# compare on exactly the same windows, or the comparison means nothing
with torch.no_grad():
    subset = {
        "naive": mae(yte[:n], xte[:n, -1:].expand(-1, H)),
        "linear (trained)": mae(yte[:n], linear(xte[:n])),
        "TimesFM zero-shot": mae(yte[:n], zero_shot),
    }
print(f"\non the same {n} test windows:")
for name, score in subset.items():
    print(f"  {name:20s} MAE {score:.4f}   ratio {score / subset['naive']:.3f}")
print(f"\n{zero_shot_secs:.0f}s for {n} windows, and NO training")
del timesfm

A model that was never shown this transformer, this county or this task beats both the
free baseline and the model we trained, in seconds. In every other lane of this course
that would be the end of the story: **use a pretrained model where you can**.

Here it is the beginning of one.

## 6.3 The check that has to come before the celebration

Before quoting a zero-shot number anywhere, ask a boring question: **was the test series
in the pretraining data?** For timeseries this is unusually likely, because the field
trains on a handful of public archives and ETTh1 is in all of them.

You cannot inspect TimesFM's mixture — its card does not document one. What you *can* do
is check whether your series appears in the public corpora that everyone builds these
mixtures from.

In [ ]:
from huggingface_hub import list_repo_files

for repo in ("Salesforce/lotsa_data", "autogluon/chronos_datasets", "Salesforce/GiftEval"):
    try:
        files = list_repo_files(repo, repo_type="dataset")
    except Exception as exc:                                  # offline, gated, renamed
        print(f"{repo:34s} could not check ({type(exc).__name__})")
        continue
    hits = sorted({f.split("/")[0] for f in files if "ett" in f.lower()})
    print(f"{repo:34s} {len(files):5d} files, ETT-looking configs: {hits if hits else 'none'}")

There it is, in the open corpora that timeseries foundation models are trained on. That
does not *prove* TimesFM saw ETTh1 — it proves that a model trained the way these models
are trained very probably did, and that nothing in the documentation lets you rule it
out.

So what is section 6.2's number worth?

- As **"TimesFM is a good forecaster for ETTh1"** — nothing. It is unfalsifiable.
- As **"a downloaded model can beat your trained one at zero marginal cost"** — still
  something, but you have not measured how much.
- As **"I should try a foundation model on my own private sensor data before building
  anything"** — quite a lot. Your data is not in anybody's corpus, and that is precisely
  where the measurement is clean.

The habit, stated once so it transfers: **a zero-shot number without a corpus check is
a number about the benchmark, not about the model.** This applies to every foundation
model you will meet — the LLM that aces a coding benchmark scraped from GitHub has the
same problem, and so does the image model evaluated on a set its LAION mixture contains.

If you need a defensible number, `Salesforce/GiftEval` exists for this: apache-2.0,
parquet-native, and explicit about train/test provenance. That is E1.

## 6.4 A result contamination cannot explain away

Set the leaderboard aside and test something structural instead — the pattern the rest
of this course is built on.

Everywhere else, we **freeze a pretrained encoder, cache its vectors, and train a small
head on top.** That is what `VectorCache` is for. `TimesFm2_5Model` (the encoder, not
`...ForPrediction`) hands us its hidden states directly: `(B, 16, 1280)`, one embedding
per 32-step patch. If the pattern holds here, a linear head on those vectors should be
at least as good as a linear layer on the raw numbers.

Contamination can only help this test — if anything, the encoder knowing ETTh1 should
make its representations *better*.

In [ ]:
encoder = TimesFm2_5Model.from_pretrained("google/timesfm-2.5-200m-transformers").eval()

STEP = 32


def embed(x: torch.Tensor) -> torch.Tensor:
    out = []
    with torch.no_grad():
        for i in range(0, len(x), STEP):
            out.append(encoder(past_values=x[i : i + STEP]).last_hidden_state)
    return torch.cat(out)                       # (N, 16, 1280)


t0 = time.time()
Etr, Ete = embed(xtr), embed(xte)
embed_secs = time.time() - t0
print(f"embedded {len(xtr) + len(xte)} windows in {embed_secs:.0f}s -> {tuple(Etr.shape)}")

In [ ]:
def probe(name: str, Ztr: torch.Tensor, Zte: torch.Tensor, epochs: int = 10) -> float:
    torch.manual_seed(0)
    head = nn.Linear(Ztr.shape[1], H)
    opt = torch.optim.Adam(head.parameters(), lr=1e-3)
    dl = DataLoader(TensorDataset(Ztr, ytr), batch_size=64, shuffle=True)
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad()
            loss_fn(head(xb), yb).backward()
            opt.step()
    with torch.no_grad():
        score = mae(yte, head(Zte))
    print(f"{name:30s} MAE {score:.4f}   ratio to naive {score / results['naive']:.3f}")
    return score


results["frozen TimesFM + head"] = probe("mean-pooled 16 patches", Etr.mean(1), Ete.mean(1))
probe("last patch only", Etr[:, -1, :], Ete[:, -1, :])
probe("embedding + raw window", torch.cat([Etr.mean(1), xtr], 1), torch.cat([Ete.mean(1), xte], 1))
probe("raw window alone", xtr, xte)

Not a small loss — a **collapse**. The frozen-encoder head is several times worse than
the naive baseline, on vectors produced by a model that, minutes ago, forecast this same
data better than anything we trained.

Three hypotheses, and the probes above kill all three:

- *Mean-pooling destroyed the ordering.* Then `last patch only` should do better. It does
  not; it is worse.
- *The head needs more than one linear layer.* Then adding the raw window alongside the
  embedding should at least match raw-alone. It does not — `embedding + raw` is **worse**
  than `raw window alone`. The embedding is not merely uninformative, it is actively in
  the way.
- *The encoder is bad.* Ruled out by 6.2: the same weights, used as intended, produced
  the best forecast in this notebook.

The pattern is not broken. It is being applied where it does not fit, and there is a
one-line test for that.

## 6.5 The rule: does the encoder compress or expand?

In [ ]:
raw_kb = xtr.element_size() * W / 1024
full_kb = Etr.element_size() * 16 * 1280 / 1024
pooled_kb = Etr.element_size() * 1280 / 1024

print(f"raw window (512 floats)      {raw_kb:6.1f} KB")
print(f"full embedding (16 x 1280)   {full_kb:6.1f} KB   ({full_kb / raw_kb:.0f}x LARGER)")
print(f"mean-pooled (1280)           {pooled_kb:6.1f} KB   ({pooled_kb / raw_kb:.1f}x larger)")
print()
print(f"for comparison, an image: 224 x 224 x 3 = {224 * 224 * 3:,} numbers -> 1280 dims, ~{224 * 224 * 3 / 1280:.0f}x SMALLER")
print(f"                a document: ~512 tokens of text        -> 384 dims")

**Rule of thumb: if an encoder expands its input, it is re-encoding, not extracting.**

That asymmetry is the whole reason freeze-and-cache works elsewhere. An image encoder
throws away 100x of pixels and keeps only what survived the pretraining objective — the
compression *is* the feature selection, and a linear head on the survivors is a
reasonable model. TimesFM's encoder goes the other way: it blows 512 numbers up 40x into
a working representation whose job is to be *decoded*, by the decoder we deleted. Take
the decoder away and you have not extracted features, you have obfuscated the input.

The test costs nothing. Before caching anything, multiply out the shapes and compare. If
the arrow points the wrong way, either use the model end to end — as 6.2 did, to good
effect — or fine-tune it. Do not freeze it and hope.

`conventions.md` states the standing rule this is evidence for: **a `VectorCache` is
used when it helps, never because it is mandated.** Here it costs 40x the storage and
several times the error. Lesson 3 therefore has no cache in it, on purpose, and this
notebook is why.

## What to take from this

- Try a pretrained timeseries model early. It is a download, it needs no training, and
  on your own data it is frequently the strongest thing available.
- **Check the pretraining corpus before quoting a zero-shot number.** If you cannot check
  it, say so when you report the number. Every published timeseries benchmark should be
  assumed contaminated until shown otherwise.
- The measurement that means something is the one on data nobody has published. That is
  usually your employer's data, which is convenient.
- Freeze-and-cache is a technique with a precondition, not a law. Compression is the
  precondition, and you can check it with a multiplication.
- A course that only ever shows its own methods succeeding has taught you a slogan. This
  one fails here, in public, with numbers.

## Exercises

**E1 — an honest benchmark.** Re-run 6.2 on a `Salesforce/GiftEval` series whose
provenance is documented, or on a series of your own that has never been published. Does
the margin over `naive` survive?

**E2 — fine-tune instead of freeze.** 6.4 froze the encoder. Unfreeze the top block and
train it with the head for two epochs. Does the pattern become viable, and at what cost
in seconds? (This is the L2 finetune-vs-frozen question, asked in a modality where the
frozen side has already lost.)

**E3 — the compression test, applied.** For each encoder you have met in this course
(MobileNet, DINOv2, the granite text embedder, TimesFM), compute input numbers over
output numbers. Sort them. Where does the freeze-and-cache pattern stop being a good
idea?

**E4 — the covariates it cannot see.** TimesFM forecast `OT` from `OT` alone; `ett.py`
also carries six load channels that physically cause it. Which of `02_basemodels`'
models can use them, and does a small model with the right inputs beat a large model
with the wrong ones?